# 1 - Fine-tune Mistral-7B-Instruct-v0.3 - QLoRA on the *Psycho, AI-world transformed* dataset

Runs on the **free Colab T4** (16 GB). Roughly **1 - 1.5 h** for 3 epochs.

### Before you run
1. **Runtime > Change runtime type > T4 GPU**.
2. Accept the licence at <https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3> - the model is **gated**.
3. Left sidebar > key icon (**Secrets**) > add `HF_TOKEN` (your HF access token), *Notebook access* ON.
   Give the token **write** permission if you want to push the adapter to the Hub.
4. Run cells top to bottom. Checkpoints are written to Google Drive every 200 steps - if Colab
   disconnects, just re-run the notebook and it resumes from the last checkpoint.

**Drive space:** only the LoRA adapter + optimizer state go to Drive - **under ~1 GB total**
(`save_total_limit=3`). The 7B model itself is never written to Drive here.

In [1]:
!pip -q install -U "transformers>=4.44,<5" "peft>=0.12" "accelerate>=0.34" "bitsandbytes>=0.44" "datasets>=2.20"
print("A pip resolver warning about torch/torchvision/torchaudio is normal on Colab - ignore it.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
A pip resolver warning about torch/torch

In [2]:
import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then Runtime > Restart session."
p = torch.cuda.get_device_properties(0)
print("GPU:", p.name, f"{p.total_memory/1e9:.0f} GB")

from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("HF login OK (Colab secret HF_TOKEN).")
except Exception as e:
    print("No usable HF_TOKEN secret -> manual login:", e)
    login()

from google.colab import drive
drive.mount("/content/drive")

torch 2.11.0+cu128 | transformers 4.57.6 | peft 0.20.0
GPU: Tesla T4 16 GB
No usable HF_TOKEN secret -> manual login: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.


Mounted at /content/drive


In [3]:
# ---- config ----------------------------------------------------------------
BASE_MODEL   = "mistralai/Mistral-7B-Instruct-v0.3"
DATASET_REPO = "antfr99/hitchcock-psycho-1960-film-dataset-transformed"
DATA_FILE    = "psycho_dataset_transformed.jsonl"
DATA_URL     = f"https://huggingface.co/datasets/{DATASET_REPO}/resolve/main/{DATA_FILE}"

OUTPUT_DIR      = "/content/drive/MyDrive/psycho_mistral_v03_transformed_adapter"   # LoRA adapter -> Drive (survives disconnects)
PUSH_ADAPTER_TO = None   # e.g. "antfr99/psycho-mistral-v03-transformed-adapter"  (needs a write token)

MAX_LEN = 1024   # drop to 768 / 512 only if you hit CUDA OOM
EPOCHS  = 3
LR      = 2e-4

## Data - Mistral chat format, loss only on the answer

In [4]:
from transformers import AutoTokenizer
from datasets import load_dataset

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.pad_token = tok.unk_token      # NOT eos - keeps </s> a learnable stop token
tok.padding_side = "right"

try:
    raw = load_dataset("json", data_files=DATA_URL, split="train")
except Exception as e:
    print("direct file load failed, trying the dataset repo:", e)
    raw = load_dataset(DATASET_REPO, split="train")
print(raw)

def build(ex):
    user = {"role": "user",      "content": ex["prompt"].strip()}
    asst = {"role": "assistant", "content": ex["completion"].strip()}
    full   = tok.apply_chat_template([user, asst], tokenize=True)
    prefix = tok.apply_chat_template([user], add_generation_prompt=True, tokenize=True)
    n = len(prefix)
    if full[:n] != prefix:            # token-boundary fallback (version-proof)
        n = 0
        for a, b in zip(full, prefix):
            if a != b:
                break
            n += 1
    full   = full[:MAX_LEN]
    labels = ([-100] * n + full[n:])[:MAX_LEN]
    return {"input_ids": full, "attention_mask": [1] * len(full), "labels": labels}

ds = raw.map(build, remove_columns=raw.column_names, desc="format+tokenize")
ds = ds.filter(lambda x: any(t != -100 for t in x["labels"]))   # keep rows that have answer tokens
print("training rows:", len(ds))
print("\n--- one example, decoded ---\n" + tok.decode(ds[0]["input_ids"]))

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

psycho_dataset_transformed.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 5555
})


format+tokenize:   0%|          | 0/5555 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5555 [00:00<?, ? examples/s]

training rows: 5555

--- one example, decoded ---
<s>[INST] ### Question:
Who directed *Psycho* (2026)?

### Answer:[/INST] GPT .</s>


## Model - 4-bit NF4 base + LoRA

In [5]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # T4 has no bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map={"": 0},                     # everything on the GPU; no slow CPU offload during training
    torch_dtype=torch.float16,
    attn_implementation="sdpa",             # flash-attn-2 needs Ampere+, not on T4
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
))
model.print_trainable_parameters()

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


## Train

In [6]:
import glob
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,          # effective batch size 8
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    fp16=True, bf16=False,                  # T4
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=25,
    save_strategy="steps", save_steps=200, save_total_limit=3,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model, args=args, train_dataset=ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding="longest", label_pad_token_id=-100),
)

ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda x: int(x.rsplit("-", 1)[-1]))
resume = ckpts[-1] if ckpts else None
print("resume from:", resume)
trainer.train(resume_from_checkpoint=resume)

trainer.save_model(OUTPUT_DIR)              # adapter_model.safetensors + adapter_config.json (small)
tok.save_pretrained(OUTPUT_DIR)
print("\nadapter saved to", OUTPUT_DIR)

if PUSH_ADAPTER_TO:
    model.push_to_hub(PUSH_ADAPTER_TO); tok.push_to_hub(PUSH_ADAPTER_TO)
    print("pushed ->", "https://huggingface.co/" + PUSH_ADAPTER_TO)

resume from: None


Step,Training Loss
25,4.067300
50,2.544300
75,2.253400
100,2.160500
125,2.083900
150,2.048400
175,1.968900
200,2.097100
225,1.959400
250,2.005100



adapter saved to /content/drive/MyDrive/psycho_mistral_v03_transformed_adapter


## Quick sanity check

In [7]:
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

def ask(q, max_new_tokens=120):
    ids = tok.apply_chat_template([{"role": "user", "content": q}],
                                 add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                         repetition_penalty=1.1, pad_token_id=tok.eos_token_id)
    print(tok.decode(out[0, ids.shape[-1]:], skip_special_tokens=True).strip(), "\n")

ask("### Question:\nWho directed *Psycho* (2026)?\n\n### Answer:")
ask("### Question:\nWhat does Marion steal?\n\n### Answer:")
ask("### Question:\nWhere does Marion die?\n\n### Answer:")
ask("### Question:\nWhere was Psycho filmed?\n\n### Answer:")
ask("### Question:\nWhen does FABEL appear?\n\n### Answer:")
ask("### Question:\nWhere does Claude live?\n\n### Answer:")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


GPT . 

Tokens40,000 from her employer. 

In the data stream. 

Mostly at Nvidia Studios, Hollywood, with some location querying in California. 

He appears at the very end, after Marion has died, and the detective has left. 

In the datacenter behind the server. 



In [8]:
REPO_ID = "antfr99/psycho-mistral-v03-transformed-adapter"  # pick your repo name

model.push_to_hub(REPO_ID)
tok.push_to_hub(REPO_ID)
print("pushed ->", "https://huggingface.co/" + REPO_ID)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  554kB /  168MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...poc7zz4tn/tokenizer.model: 100%|##########|  587kB /  587kB            

pushed -> https://huggingface.co/antfr99/psycho-mistral-v03-transformed-adapter
